<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Session 5 — A Practical Introduction to Machine Learning</h2>
</div>

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Learning Goals</h2>
<ul>
<li>Understand the difference between supervised and unsupervised machine learning</li>
<li>Explain what kinds of questions classification and regression can answer</li>
<li>Fit and interpret logistic regression, KNN, decision trees, and random forests</li>
<li>Understand clustering and dimensionality reduction through K-means and PCA</li>
<li>Connect machine learning techniques to concrete social science questions</li>
</ul>
</div>

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why machine learning matters in social science</h2>
Machine learning gives us a set of tools for finding patterns, making predictions, and organizing complex data. In social science, these tools can help answer questions such as:<ul><li>Which students are at risk of dropping out?</li><li>Which survey respondents are likely to vote?</li><li>What predicts support for a policy?</li><li>Are there hidden groups of people with similar attitudes or behaviors?</li></ul>Machine learning is not magic. It does not replace theory, context, or careful interpretation. But it can help us ask better questions and analyze larger or more complex datasets.
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
np.random.seed(42)

## 1. The Big Picture

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Supervised vs. Unsupervised Learning</h2>
<b>Supervised learning</b> uses labeled data. That means we already know the outcome we want to predict. The model learns the relationship between predictors and that known outcome.<br><br>
Examples:<ul><li>Predict whether a student will graduate</li><li>Predict a person's income</li><li>Predict whether someone voted in the last election</li></ul>
<b>Unsupervised learning</b> uses unlabeled data. There is no known outcome column. Instead, the model tries to discover structure in the data.<br><br>
Examples:<ul><li>Find groups of neighborhoods with similar demographic profiles</li><li>Identify survey respondents with similar attitudes</li><li>Reduce many variables into a smaller number of dimensions</li></ul>
</div>

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Which of the following is supervised learning, and which is unsupervised?
<ul>
<li>Predicting whether a student drops a course</li>
<li>Grouping cities based on housing, crime, and income statistics</li>
<li>Predicting annual salary from education and work experience</li>
</ul>
Write your answers in the code cell below as comments.
</div>

In [ ]:
# Your answers here


## 2. Build Example Datasets

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Our teaching datasets</h2>
To make the notebook self-contained, we will generate synthetic data with social science themes. This lets us focus on concepts without needing extra files.<br><br>
We will use three datasets:<ul><li><b>Student retention data</b> for classification models</li><li><b>Policy support data</b> for regression</li><li><b>Survey response data</b> for clustering and PCA</li></ul>
</div>

In [ ]:
n = 300

# Classification dataset: student retention
study_hours = np.random.normal(12, 3, n).clip(2, 25)
attendance = np.random.normal(82, 10, n).clip(40, 100)
stress = np.random.normal(5.5, 1.8, n).clip(1, 10)
financial_pressure = np.random.normal(5, 2, n).clip(1, 10)
support_services = np.random.normal(4, 2, n).clip(0, 10)

logit = -6 + 0.09 * attendance + 0.18 * study_hours - 0.25 * stress - 0.18 * financial_pressure + 0.22 * support_services
prob_retained = 1 / (1 + np.exp(-logit))
retained = np.where(np.random.rand(n) < prob_retained, 1, 0)

students = pd.DataFrame({
    "study_hours": study_hours,
    "attendance": attendance,
    "stress": stress,
    "financial_pressure": financial_pressure,
    "support_services": support_services,
    "retained": retained
})

# Regression dataset: policy support score
education_years = np.random.normal(15, 2.5, n).clip(8, 22)
media_exposure = np.random.normal(6, 2, n).clip(0, 12)
political_interest = np.random.normal(5.5, 2, n).clip(0, 10)
community_trust = np.random.normal(5.8, 1.8, n).clip(0, 10)
age = np.random.normal(39, 12, n).clip(18, 80)
noise = np.random.normal(0, 6, n)

policy_support = (
    25
    + 1.4 * education_years
    + 2.2 * political_interest
    + 1.1 * community_trust
    + 0.6 * media_exposure
    - 0.08 * age
    + noise
).clip(0, 100)

policy = pd.DataFrame({
    "education_years": education_years,
    "media_exposure": media_exposure,
    "political_interest": political_interest,
    "community_trust": community_trust,
    "age": age,
    "policy_support": policy_support
})

# Unsupervised dataset: attitude survey
group = np.random.choice([0, 1, 2], size=n, p=[0.35, 0.35, 0.30])
trust_gov = np.where(group == 0, np.random.normal(8, 1, n), np.where(group == 1, np.random.normal(4, 1, n), np.random.normal(6, 1, n))).clip(1, 10)
trust_media = np.where(group == 0, np.random.normal(7, 1.2, n), np.where(group == 1, np.random.normal(3.5, 1, n), np.random.normal(5.5, 1, n))).clip(1, 10)
civic_engagement = np.where(group == 0, np.random.normal(7.5, 1, n), np.where(group == 1, np.random.normal(4, 1, n), np.random.normal(6, 1, n))).clip(1, 10)
social_trust = np.where(group == 0, np.random.normal(7, 1, n), np.where(group == 1, np.random.normal(4.2, 1, n), np.random.normal(5.8, 1, n))).clip(1, 10)
economic_anxiety = np.where(group == 0, np.random.normal(3.5, 1, n), np.where(group == 1, np.random.normal(7.5, 1, n), np.random.normal(5.5, 1, n))).clip(1, 10)

survey = pd.DataFrame({
    "trust_gov": trust_gov,
    "trust_media": trust_media,
    "civic_engagement": civic_engagement,
    "social_trust": social_trust,
    "economic_anxiety": economic_anxiety
})

students.head(), policy.head(), survey.head()

## 3. Supervised Learning

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">What question can supervised learning answer?</h2>
Supervised learning answers questions where we already know the target outcome in our training data. The model learns from examples and then predicts outcomes for new cases.<br><br>
<b>Example social science question:</b> Can we predict whether a student will stay enrolled based on attendance, study hours, stress, and support?
</div>

### Classification vs. Regression

- **Classification** predicts categories such as `yes/no`, `graduate/dropout`, or `voted/did not vote`.
- **Regression** predicts numeric values such as income, support score, or medical cost.

## 4. Logistic Regression

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Logistic regression</h2>
Logistic regression is used when the outcome is binary, such as `0/1` or `yes/no`. Instead of predicting a raw number, it predicts a probability and then converts that probability into a class label.<br><br>
<b>Question it can answer:</b> What is the probability that a student will be retained, given their attendance, study hours, stress, and support services?
</div>

In [ ]:
X = students[["study_hours", "attendance", "stress", "financial_pressure", "support_services"]]
y = students["retained"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

log_model.fit(X_train, y_train)
log_preds = log_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, log_preds), 3))
print(confusion_matrix(y_test, log_preds))

### Interpretation

If accuracy is high, the model is doing a decent job of separating retained and not-retained students. In a real educational setting, this kind of model could help identify students who may need support, but only if used carefully and ethically.

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Print a classification report for the logistic regression model. Then write one or two sentences explaining what accuracy means in this context.
</div>

In [ ]:
# Your code here


## 5. K-Nearest Neighbors (KNN)

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">K-Nearest Neighbors</h2>
KNN classifies a new case by looking at the most similar observations nearby. If most nearby observations belong to one class, the new case is assigned to that class.<br><br>
<b>Question it can answer:</b> If a student looks similar to other retained students in terms of attendance, stress, and study habits, are they also likely to be retained?
</div>

In [ ]:
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=7))
])

knn_model.fit(X_train, y_train)
knn_preds = knn_model.predict(X_test)

print("KNN Accuracy:", round(accuracy_score(y_test, knn_preds), 3))

### Why scaling matters for KNN

KNN depends on distances. If one variable has a much larger scale than the others, it can dominate the distance calculation. That is why standardization is usually important for KNN.

## 6. Decision Trees

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Decision trees</h2>
A decision tree splits the data into branches based on rules. It asks a series of questions such as: Is attendance above 80? Is stress below 6? Each split helps separate the classes.<br><br>
<b>Question it can answer:</b> What combinations of student characteristics are associated with staying enrolled?
</div>

In [ ]:
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)
tree_preds = tree_model.predict(X_test)

print("Decision Tree Accuracy:", round(accuracy_score(y_test, tree_preds), 3))

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(tree_model, feature_names=X.columns, class_names=["Not Retained", "Retained"], filled=True, rounded=True)
plt.show()

### Why social science students often like decision trees

Decision trees are easy to interpret visually. They make the model feel concrete because you can literally see the decision rules. That interpretability is often valuable in public policy, education, and sociology.

## 7. Random Forest

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Random forests</h2>
A random forest combines many decision trees. Each tree sees a slightly different sample of the data and a slightly different set of predictors. The final prediction is based on the combined votes of many trees.<br><br>
<b>Question it can answer:</b> Can we improve prediction of student retention by averaging across many different trees instead of relying on a single one?
</div>

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("Random Forest Accuracy:", round(accuracy_score(y_test, rf_preds), 3))

importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
importance

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x=importance.values, y=importance.index, color="#2563eb")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Predictor")
plt.show()

### Why random forests are useful

Random forests often predict better than a single decision tree because they reduce overfitting. The tradeoff is that they are less transparent. You gain predictive power, but lose some interpretability.

## 8. Regression for Continuous Outcomes

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Linear regression</h2>
When the outcome is numeric rather than categorical, we use regression models for continuous prediction.<br><br>
<b>Question it can answer:</b> How strongly do education, political interest, trust, and media exposure predict support for a policy?
</div>

In [ ]:
X_reg = policy[["education_years", "media_exposure", "political_interest", "community_trust", "age"]]
y_reg = policy["policy_support"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.25, random_state=42)

lin_model = LinearRegression()
lin_model.fit(Xr_train, yr_train)
reg_preds = lin_model.predict(Xr_test)

coef_table = pd.DataFrame({
    "predictor": X_reg.columns,
    "coefficient": lin_model.coef_
}).round(3)

print("R-squared:", round(r2_score(yr_test, reg_preds), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(yr_test, reg_preds)), 3))
coef_table

### Interpretation

If the coefficient for political interest is positive, then higher political interest is associated with higher policy support, holding the other predictors constant. In social science, that phrase “holding the other predictors constant” is very important.

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Which predictor seems to have the largest positive relationship with policy support? Which one seems negative? Write a short interpretation below.
</div>

In [ ]:
# Your interpretation here


## 9. Unsupervised Learning

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">What question can unsupervised learning answer?</h2>
Unsupervised learning is useful when there is no target column to predict. Instead, the goal is to uncover structure in the data.<br><br>
<b>Example social science question:</b> Are there distinct types of survey respondents based on trust, engagement, and economic anxiety?
</div>

## 10. K-Means Clustering

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">K-means clustering</h2>
K-means clustering groups observations into clusters so that cases within the same cluster are similar to one another. The analyst chooses the number of clusters, <code>k</code>.<br><br>
<b>Question it can answer:</b> Can we identify different profiles of respondents based on trust in institutions, civic engagement, and economic anxiety?
</div>

In [ ]:
scaler = StandardScaler()
survey_scaled = scaler.fit_transform(survey)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(survey_scaled)

survey_clustered = survey.copy()
survey_clustered["cluster"] = clusters
survey_clustered.groupby("cluster").mean().round(2)

### Interpretation

Once we compute cluster averages, we can describe the groups in substantive language. For example, one cluster may look like highly engaged and institutionally trusting respondents, while another may look more anxious and less trusting.

## 11. Principal Component Analysis (PCA)

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Principal Component Analysis</h2>
PCA is a dimensionality reduction technique. It takes many variables and summarizes them into a smaller number of components that capture most of the variation in the data.<br><br>
<b>Question it can answer:</b> If we have many survey items, can we reduce them to a smaller set of underlying dimensions such as trust or civic orientation?
</div>

In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(survey_scaled)

pca_df = pd.DataFrame({
    "PC1": components[:, 0],
    "PC2": components[:, 1],
    "cluster": clusters
})

print("Explained variance ratio:", pca.explained_variance_ratio_.round(3))
pd.DataFrame(pca.components_, columns=survey.columns, index=["PC1", "PC2"]).round(3)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="cluster", palette="deep")
plt.title("Survey Respondents Projected onto Two Principal Components")
plt.show()

### Interpretation

PCA helps us visualize complex data in a smaller space. If respondents separate clearly in the first two components, that suggests the survey responses have meaningful lower-dimensional structure.

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Look at the cluster means and PCA plot. How would you describe the clusters in plain social science language? Write a few sentences below.
</div>

In [ ]:
# Your interpretation here


## 12. Comparing Techniques

| Technique | Type | Target Variable | Example Social Science Question |
|---|---|---|---|
| Logistic Regression | Supervised | Binary labeled target | Will a student remain enrolled? |
| KNN | Supervised | Binary or multi-class labeled target | Which group does this case most resemble? |
| Decision Tree | Supervised | Binary or multi-class labeled target | What decision rules separate different outcomes? |
| Random Forest | Supervised | Binary or multi-class labeled target | Can we improve prediction by combining many trees? |
| Linear Regression | Supervised | Continuous labeled target | What predicts a support score or income? |
| K-Means | Unsupervised | No labeled target | Are there distinct types of respondents? |
| PCA | Unsupervised | No labeled target | Can many variables be reduced to a few dimensions? |

**Note:** In unsupervised learning, there is no labeled target variable to predict. That does not mean the method has no output. Instead, the model discovers structure in the data. For example, **K-Means** outputs cluster assignments for each observation and estimates cluster centers, while **PCA** outputs principal components and the proportion of variance each component explains.

## 13. Ethics and Caution

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Important caution</h2>
Machine learning models can reflect bias in the training data. In social science and education, these models may affect people in high-stakes contexts. That means prediction alone is never enough. We also need theory, fairness checks, domain knowledge, and ethical judgment.
</div>

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Final Practice</h2>
Choose one social science question of your own. Then answer:
<ol>
<li>Would you use supervised or unsupervised learning?</li>
<li>Which specific technique from this notebook would you choose?</li>
<li>Why is that technique a good fit for your question?</li>
</ol>
</div>

In [ ]:
# Your response here
